# Timeline + Donut Chart Development Notebook

This notebook is for iterative development and testing of:
- `player_full_profile_career_timeline_chart`
- `player_full_profile_position_donut_chart`

It helps you regenerate dev chart specs, inspect structure, filter to a single player, and render component layers in isolation.

## 1. Setup

In [2]:
from pathlib import Path
import json
import sys
import importlib
from datetime import datetime

import altair as alt
import pandas as pd

# Resolve project root robustly even if notebook is moved
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'pyproject.toml').exists():
            PROJECT_ROOT = parent
            break

PYTHON_DIR = PROJECT_ROOT / 'python'
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

print('Project root:', PROJECT_ROOT)
print('Python dir  :', PYTHON_DIR)
print('Loaded at   :', datetime.now().isoformat(timespec='seconds'))

Project root: /Users/samlindsay/Documents/Projects/Personal/EGRFC/egrfc-stats
Python dir  : /Users/samlindsay/Documents/Projects/Personal/EGRFC/egrfc-stats/python
Loaded at   : 2026-04-22T16:52:59


In [3]:
import backend as backend_module
import charts as charts_module

# Keep references to modules so we can reload them after edits.
BackendDatabase = backend_module.BackendDatabase

DB_PATH = PROJECT_ROOT / 'data' / 'egrfc_backend.duckdb'
db = BackendDatabase()
db.build()
print('Database initialized from', DB_PATH)

INFO:root:Loaded 733 matches from consolidated file


Removed 8 unfulfilled Pitchero fixtures (0-0 or non-numeric scores)
Removing 1 Pitchero-only rows overlapping with Google games.
Database initialized from /Users/samlindsay/Documents/Projects/Personal/EGRFC/egrfc-stats/data/egrfc_backend.duckdb


In [9]:
def clean_name(name):

    name_dict = {
        "Sam Lindsay": "S Lindsay 2",
        "Sam Lindsay-McCall": "S Lindsay",
        "James Mitchell": "T Mitchell",
    }

    if name in name_dict:
        return name_dict[name]

    initial = name.split(" ")[0][0]
    surname = " ".join(name.split(" ")[1:])
    surname = surname.replace("’", "'")
    name_clean = f"{initial} {surname}"
    # trim and title case
    return name_clean.strip().title()

def _parse_profile_scorer_payload(value):
    if value is None:
        return {}
    if isinstance(value, dict):
        return {
            str(key).strip(): int(count)
            for key, count in value.items()
            if str(key).strip() and pd.notna(count)
        }

    raw = str(value or '').strip()
    if not raw or raw in {'{}', '[]', 'null', 'None'}:
        return {}

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return {}

    if not isinstance(parsed, dict):
        return {}

    payload = {}
    for key, count in parsed.items():
        name = str(key).strip()
        if not name:
            continue
        try:
            payload[name] = int(count)
        except (TypeError, ValueError):
            continue
    return payload

In [47]:
appearances_df = db.con.execute(
    """
    SELECT
        pa.player,
        pa.squad,
        pa.date,
        pa.is_captain,
        pa.club_appearance_number,
        g.opposition,
        g.home_away,
        g.game_id,
        g.result,
        g.score_for,
        g.score_against
    FROM player_appearances pa
    LEFT JOIN games g USING (game_id)
    ORDER BY pa.player, pa.date, pa.game_id
    """
).df()

games_df = db.con.execute(
    """
    SELECT
        game_id, date, opposition, home_away,
        tries_scorers, motm, result, score_for, score_against
    FROM games
    ORDER BY date, game_id
    """
).df()

milestone_counts = {25, 50, 100}
rows = []

if not appearances_df.empty:
    appearances_df['date'] = pd.to_datetime(appearances_df['date'], errors='coerce')
    appearances_df = appearances_df.dropna(subset=['date']).copy()
if not games_df.empty:
    games_df['date'] = pd.to_datetime(games_df['date'], errors='coerce')
    games_df = games_df.dropna(subset=['date']).copy()
def _fmt_result(result, sf, sa):
    if not result or pd.isna(result):
        return ''
    letter = str(result)[0].upper()
    try:
        return f"{letter} {int(sf)}-{int(sa)}"
    except (TypeError, ValueError):
        return letter
for player, player_apps in appearances_df.groupby('player', sort=False):
    player_apps = player_apps.sort_values(['date', 'game_id']).reset_index(drop=True)
    if player_apps.empty:
        continue
    player_key = clean_name(player)
    player_game_ids = set(player_apps['game_id'])
    player_1st_game_ids = set(player_apps[player_apps['squad'].astype(str).eq('1st')]['game_id'])
    def add_event(event_key, label, row_type, date_value, detail,
                  squad='', game_id='', result='', score_for=None, score_against=None):
        dt = pd.to_datetime(date_value, errors='coerce')
        if pd.isna(dt):
            return
        rows.append({
            'player': player,
            'event_key': event_key,
            'event_label': label,
            'row_type': row_type,
            'date': dt,
            'detail': detail,
            'squad': str(squad) if squad and pd.notna(squad) else '',
            'game_id': game_id,
            'result_label': _fmt_result(result, score_for, score_against),
            'event_rank': len(rows) + 1,
        })
    # ── Appearance milestones ──────────────────────────────────────────
    debut = player_apps.iloc[0]
    add_event('debut', 'Club debut', 'appearance',
              debut['date'], f"v {debut['opposition']} ({debut['home_away']})",
              debut['squad'], debut['game_id'],
              debut.get('result'), debut.get('score_for'), debut.get('score_against'))
    first_xv_apps = player_apps[player_apps['squad'].astype(str).eq('1st')].reset_index(drop=True)
    if not first_xv_apps.empty:
        xv_debut = first_xv_apps.iloc[0]
        if xv_debut['game_id'] != debut['game_id']:
            add_event('first_xv_debut', '1st XV debut', 'appearance',
                      xv_debut['date'], f"v {xv_debut['opposition']} ({xv_debut['home_away']})",
                      xv_debut['squad'], xv_debut['game_id'],
                      xv_debut.get('result'), xv_debut.get('score_for'), xv_debut.get('score_against'))
    # Club appearance milestones (25, 50, 100)
    club_milestone_game_ids = {}
    for _, app_row in player_apps.iterrows():
        try:
            n = int(app_row['club_appearance_number'])
        except (TypeError, ValueError):
            continue
        if n in milestone_counts:
            club_milestone_game_ids[n] = app_row['game_id']
            add_event(f'milestone_{n}', f'{n} appearances', 'appearance',
                      app_row['date'], f"v {app_row['opposition']} ({app_row['home_away']})",
                      app_row['squad'], app_row['game_id'],
                      app_row.get('result'), app_row.get('score_for'), app_row.get('score_against'))
    # 1st XV appearance milestones — skip if coincides with club milestone
    for xv_idx, xv_row in first_xv_apps.iterrows():
        n = xv_idx + 1
        if n in milestone_counts and club_milestone_game_ids.get(n) != xv_row['game_id']:
            add_event(f'milestone_{n}_1st', f'{n} 1st XV appearances', 'appearance',
                      xv_row['date'], f"v {xv_row['opposition']} ({xv_row['home_away']})",
                      xv_row['squad'], xv_row['game_id'],
                      xv_row.get('result'), xv_row.get('score_for'), xv_row.get('score_against'))
    last_row = player_apps.iloc[-1]
    add_event('last_game', 'Last appearance', 'appearance',
              last_row['date'], f"v {last_row['opposition']} ({last_row['home_away']})",
              last_row['squad'], last_row['game_id'],
              last_row.get('result'), last_row.get('score_for'), last_row.get('score_against'))
    # ── Event milestones ───────────────────────────────────────────────
    player_games_sorted = (
        games_df[games_df['game_id'].isin(player_game_ids)]
        .sort_values('date')
        .copy()
    )
    # Try milestones
    first_try_game = None
    first_xv_try_game = None
    for _, game in player_games_sorted.iterrows():
        payload = _parse_profile_scorer_payload(game.get('tries_scorers'))
        is_scorer = any(clean_name(nm) == player_key and c > 0 for nm, c in payload.items())
        if is_scorer:
            if first_try_game is None:
                first_try_game = game
            if first_xv_try_game is None and game['game_id'] in player_1st_game_ids:
                first_xv_try_game = game
        if first_try_game is not None and first_xv_try_game is not None:
            break
    if first_try_game is not None:
        g = first_try_game
        add_event('first_try', 'First try', 'event',
                  g['date'], f"v {g['opposition']} ({g['home_away']})",
                  '', g['game_id'], g.get('result'), g.get('score_for'), g.get('score_against'))
    if first_xv_try_game is not None and (
        first_try_game is None or first_xv_try_game['game_id'] != first_try_game['game_id']
    ):
        g = first_xv_try_game
        add_event('first_xv_try', 'First 1st XV try', 'event',
                  g['date'], f"v {g['opposition']} ({g['home_away']})",
                  '1st', g['game_id'], g.get('result'), g.get('score_for'), g.get('score_against'))
    # Captaincy milestones
    cap_rows = player_apps[player_apps['is_captain'] == True]
    if not cap_rows.empty:
        cap = cap_rows.iloc[0]
        add_event('first_captaincy', 'First captaincy', 'event',
                  cap['date'], f"v {cap['opposition']} ({cap['home_away']})",
                  cap['squad'], cap['game_id'],
                  cap.get('result'), cap.get('score_for'), cap.get('score_against'))
    # MOTM (all occurrence)
    for _, game in player_games_sorted.iterrows():
        motm_val = game.get('motm')
        if motm_val and pd.notna(motm_val) and clean_name(str(motm_val)) == player_key:
            g = game
            add_event('motm', 'Man of the Match', 'event',
                      g['date'], f"v {g['opposition']} ({g['home_away']})",
                      '', g['game_id'], g.get('result'), g.get('score_for'), g.get('score_against'))
# ── Build timeline DataFrame ──────────────────────────────────────────
timeline_df = pd.DataFrame(rows)
if timeline_df.empty:
    timeline_df = pd.DataFrame(columns=[
        'player', 'event_key', 'event_label', 'row_type', 'date',
        'detail', 'squad', 'game_id', 'result_label', 'event_rank',
        'event_date_label', 'event_sort', 'squad_label', 'row_y',
        'symbol_text', 'symbol_fill', 'symbol_stroke', 'symbol_text_color',
    ])
else:
    timeline_df = timeline_df.sort_values(['player', 'date', 'event_rank']).reset_index(drop=True)
    timeline_df['event_date_label'] = timeline_df['date'].dt.strftime('%-d %b %Y')
    timeline_df['event_sort'] = timeline_df['date'].astype('int64')
    timeline_df['squad_label'] = timeline_df['squad'].map(
        {'1st': '1st XV', '2nd': '2nd XV'}
    ).fillna('Club')
    # row_y: 1 = appearance track (lower, near x-axis), 0 = event track (upper)
    timeline_df['row_y'] = timeline_df['row_type'].map({'appearance': 1, 'event': 0}).fillna(0)
    event_style_map = {
        # ── Appearance milestones ─────────────────────────────────────
        'debut':             {'symbol_text': '1',   'fill': '#ffffff', 'stroke': '#202946', 'text_color': '#202946'},
        'first_xv_debut':    {'symbol_text': '1',   'fill': '#202946', 'stroke': '#7d96e8', 'text_color': '#ffffff'},
        'milestone_25':      {'symbol_text': '25',  'fill': '#c6894a', 'stroke': 'transparent',    'text_color': '#3d230f'},
        'milestone_50':      {'symbol_text': '50',  'fill': '#b9c2cf', 'stroke': 'transparent',    'text_color': '#1f2833'},
        'milestone_100':     {'symbol_text': '100', 'fill': '#c6894a', 'stroke': 'transparent',    'text_color': '#4a3500'},
        'milestone_25_1st':  {'symbol_text': '25',  'fill': '#c6894a', 'stroke': '#9f6b37', 'text_color': '#3d230f'},
        'milestone_50_1st':  {'symbol_text': '50',  'fill': '#b9c2cf', 'stroke': '#8e98a8', 'text_color': '#1f2833'},
        'milestone_100_1st': {'symbol_text': '100', 'fill': '#c6894a', 'stroke': '#d39d00', 'text_color': '#4a3500'},
        'last_game':         {'symbol_text': '',    'fill': '#ffffff', 'stroke': '#000000', 'text_color': '#000000'},
        # ── Event milestones ──────────────────────────────────────────
        'first_try':         {'symbol_text': 'T',  'fill': '#991515', 'stroke': 'transparent',    'text_color': '#991515'},
        'first_xv_try':      {'symbol_text': 'T',  'fill': '#991515', 'stroke': 'transparent',    'text_color': '#991515'},
        'first_captaincy':   {'symbol_text': 'C',  'fill': '#7d96e8', 'stroke': 'transparent',    'text_color': '#7d96e8'},
        'motm':              {'symbol_text': '',    'fill': '#c6894a', 'stroke': '#202946', 'text_color': '#c6894a'},
    }
    timeline_df['symbol_text']       = timeline_df['event_key'].map(lambda k: event_style_map.get(k, {}).get('symbol_text', ''))
    timeline_df['symbol_fill']       = timeline_df['event_key'].map(lambda k: event_style_map.get(k, {}).get('fill', '#202946'))
    timeline_df['symbol_stroke']     = timeline_df['event_key'].map(lambda k: event_style_map.get(k, {}).get('stroke', 'none'))
    timeline_df['symbol_text_color'] = timeline_df['event_key'].map(lambda k: event_style_map.get(k, {}).get('text_color', ''))
    timeline_df['symbol_shape']      = timeline_df['event_key'].map(lambda k: 'M-.5206-6.2483-.3929-6.5554-.277-6.834C-.1745-7.0804.1745-7.0804.277-6.834L.3929-6.5554.5206-6.2483.5415-6.198 1.9134-2.8996C1.9566-2.7957 2.0543-2.7248 2.1664-2.7158L5.7274-2.4303 5.7816-2.426 6.1132-2.3994 6.414-2.3753C6.68-2.354 6.7878-2.022 6.5852-1.8484L6.356-1.6521 6.1034-1.4357 6.062-1.4003 3.349.9237C3.2636.9969 3.2263 1.1117 3.2524 1.2211L4.0812 4.6959 4.0939 4.7489 4.171 5.0724 4.2411 5.3659C4.303 5.6255 4.0206 5.8307 3.7929 5.6916L3.5354 5.5343 3.2515 5.3609 3.205 5.3325.1564 3.4704C.0604 3.4118-.0603 3.4118-.1563 3.4704L-3.2049 5.3325-3.2514 5.3609-3.5352 5.5343-3.7927 5.6916C-4.0204 5.8307-4.3028 5.6255-4.2409 5.3659L-4.1709 5.0724-4.0937 4.7489-4.0811 4.6959-3.2522 1.2211C-3.2261 1.1117-3.2634.9968-3.3488.9237L-6.0618-1.4003-6.1032-1.4357-6.3558-1.6521-6.585-1.8484C-6.7877-2.022-6.6798-2.354-6.4138-2.3753L-6.113-2.3994-5.7815-2.426-5.7272-2.4303-2.1663-2.7158C-2.0542-2.7248-1.9565-2.7958-1.9133-2.8996L-.5414-6.198-.5205-6.2483Z' if k == 'motm' else 'circle')   

# ── Chart ─────────────────────────────────────────────────────────────
tooltip_encoding = [
    alt.Tooltip('player:N',           title='Player'),
    alt.Tooltip('event_label:N',      title='Milestone'),
    alt.Tooltip('event_date_label:N', title='Date'),
    alt.Tooltip('detail:N',           title='Match'),
    alt.Tooltip('result_label:N',     title='Result'),
    alt.Tooltip('squad_label:N',      title='Squad'),
]
year_axis = alt.Axis(
    format='%Y',
    grid=False,
    labelAngle=0,
    labelFlush=False,
    tickCount='year',
    tickSize=10,
    labelFontSize=18,
    domainWidth=3,
    tickWidth=2,
    title=None
)
# y scale: row_y=1 → appearance track (lower), row_y=0 → event track (upper)
y_enc   = alt.Y(
    'row_type:N', 
    scale=alt.Scale(
        domain=['event', 'appearance'], 
        range=[0, 1]
    ), 
    axis=alt.Axis(title=None, values=['event', 'appearance'], labels=False, ticks=False, domain=False)  
)

source = alt.Chart(timeline_df).transform_filter(alt.datum.player == 'Rory Evans').encode(
    x=alt.X('date:T', axis=year_axis),
    y=alt.Y('row_type:N', axis=None),
    tooltip=tooltip_encoding,
)

# Connecting line through appearance milestones (debut → last_game)
connecting_line = (
    source
    .transform_filter(alt.datum.row_type == 'appearance')
    .mark_line(color='#202946', strokeWidth=5, opacity=0.1)
)
# Circles for appearance milestones
appearance_circles = (
    source
    .transform_filter(alt.datum.row_type == 'appearance')
    .mark_point(filled=True, shape='circle', size=600, strokeWidth=2.2, opacity=1)
    .encode(
        color=alt.Color('symbol_fill:N', scale=None, legend=None),
        stroke=alt.Stroke('symbol_stroke:N', scale=None, legend=None)
    )
)
# Numbers/label inside circles
circle_text = (
    source
    .transform_filter(
        (alt.datum.row_type == 'appearance') & (alt.datum.symbol_text != '')
    )
    .mark_text(
        align='center', baseline='middle',
        font='PT Sans Narrow', fontWeight='bold', fontSize=13,
    )
    .encode(
        text=alt.Text('symbol_text:N'),
        color=alt.Color('symbol_text_color:N', scale=None, legend=None),
    )
)
# Bold letter glyphs for event milestones (T, C)
event_letters = (
    source
    .transform_filter(
        (alt.datum.row_type == 'event') & (alt.datum.event_key != 'motm')
    )
    .mark_text(
        align='center', baseline='middle',
        font='PT Sans Narrow', fontWeight='bold', fontSize=20,
    )
    .encode(
        text=alt.Text('symbol_text:N'),
        color=alt.Color('symbol_text_color:N', scale=None, legend=None),
    )
)

# Star for MOTM
motm_stars = (
    source
    .transform_filter(alt.datum.event_key == 'motm')
    .mark_point(size=12, strokeWidth=1, opacity=1)
    .encode(
        fill=alt.Color('symbol_fill:N', scale=None, legend=None),
        stroke=alt.Stroke('symbol_stroke:N', scale=None, legend=None),
        shape=alt.Shape('symbol_shape:N', scale=None, legend=None)

    )
)
# Diagonal detail labels (match reference, e.g. "v Horsham (A)")
detail_text = (
    source
    # .transform_filter(alt.datum.row_type == 'appearance')
    .mark_text(
        align='left', baseline='bottom', angle=300, color='#4a5568',
        font='PT Sans Narrow', fontSize=12,
        yOffset=-15, xOffset=15,
    )
    .encode(
        text=alt.Text('detail:N'),
        color=alt.Color('symbol_text_color:N', scale=None, legend=None)
    )
)
chart = (
    alt.layer(
        connecting_line, 
        appearance_circles, 
        circle_text,
        event_letters, 
        motm_stars, 
        detail_text,
    )
    .properties(width=750, height=60)
    .resolve_scale(y='shared', x='shared', shape='independent')
    .configure_view(strokeWidth=0)
)

chart

NameError: name 'clean_name' is not defined

In [76]:
from python.chart_helpers import alt_theme

alt.themes.register('custom_theme', alt_theme)
alt.themes.enable('custom_theme')

df = db.con.execute(
    """
    SELECT
        player,
        NULLIF(position, '') AS position,
        COUNT(*) AS appearances,
    FROM player_appearances
    WHERE player = 'Rory Evans'
    GROUP BY 1, 2
    """
).df()

df['position'] = df['position'].astype(str).str.strip()
totals = (
    df.groupby('position', as_index=False)['appearances']
    .sum()
    .sort_values(['appearances', 'position'], ascending=[False, True])
    .reset_index(drop=True)
)
sort_order_map = {row['position']: idx for idx, row in totals.iterrows()}
df['sort_order'] = df['position'].map(sort_order_map).fillna(999).astype(int)

palette_by_rank = ['#202946', '#7d96e8', '#146f14', '#c48b14', '#981515', '#5b657d']
bench_color = '#8b93a7'

regular_positions = [
    position for position in df.sort_values('sort_order')['position'].unique().tolist()
    if position.lower() != 'bench'
]
color_domain = regular_positions.copy()
color_range = [palette_by_rank[min(idx, len(palette_by_rank) - 1)] for idx, _ in enumerate(regular_positions)]
if any(df['position'].str.lower() == 'bench'):
    color_domain.append('Bench')
    color_range.append(bench_color)

base = alt.Chart(df).encode(
    order=alt.Order('appearances:Q', sort='ascending'),
    theta=alt.Theta('appearances:Q', title='Appearances', sort="descending", stack=True)
)

arc = (
    base
    .mark_arc(innerRadius=40, outerRadius=120, stroke='#ffffff', strokeWidth=2)
    .encode(
        color=alt.Color(
            'position:N',
            sort=alt.EncodingSortField(field='sort_order', op='min', order='ascending'),
            scale=alt.Scale(domain=color_domain, range=color_range),
            legend=None,
        ),
        tooltip=[
            alt.Tooltip('player:N', title='Player'),
            alt.Tooltip('position:N', title='Position'),
            alt.Tooltip('appearances:Q', title='Appearances'),
        ],
    )
)

position_labels = base.mark_text(
    font='PT Sans Narrow',
    fontSize=18,
    fontWeight='bold',
    radius=150,
    dy=-5
).encode(
    color=alt.Color('position:N', scale=alt.Scale(domain=color_domain, range=color_range), legend=None),
    text=alt.Text('position_label:N'),
    opacity=alt.condition(alt.datum.appearances > 2, alt.value(1), alt.value(0))
).transform_calculate(
    position_label="split(datum.position, ' ')"
)

count_labels = base.mark_text(
    fontSize=18,
    color='white',
    radius=80
).encode(
    text=alt.Text('appearances:N')
)

chart = (arc + position_labels + count_labels).resolve_scale(color='shared').configure_view(strokeWidth=0, fill=None).properties(title="Position").configure(background="#e5e4e7")

chart


alt.LayerChart(...)